In [1]:
import os, sys
from tqdm import tqdm
import torch
import numpy as np

sys.path.insert(0, os.path.join(os.path.abspath(''), '..'))
from cmm.ffxml import ForceFieldXML
from cmm.topology import Topology
from cmm.units import BOHR2NM, BOHR2ANG, HARTREE2KCAL

import openmm.app as app
from ase.io import read

In [2]:
torch.set_default_dtype(torch.float64)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
ff_path = os.path.join(os.path.abspath(''), '../scripts/water_refit.xml')
ff = ForceFieldXML(ff_path, device=device)



In [3]:
water_cluster_path = os.path.join(os.path.abspath(''), '../tests/data/water_clusters/all_reference_clusters.pdb')

In [7]:
water_cluster_pdb = app.PDBFile(water_cluster_path)
positions = [water_cluster_pdb.getPositions(True, frame=i)._value * 10.0 for i in range(water_cluster_pdb.getNumFrames())]

topologies = [Topology.fromMultiPDB(water_cluster_path, device, frame_index=i) for i in range(water_cluster_pdb.getNumFrames())]
systems = [ff.parametrize(topology, use_fd_morse=True, periodic=False, use_lr_dispersion=False, use_cutoff=False) for topology in topologies]

In [8]:
coords = torch.from_numpy(positions[-1] / BOHR2ANG).to(device).requires_grad_(False)
natoms = coords.shape[-1]
box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
energies = systems[-1].getEnergy(coords, box)

for key in energies:
    print(f"{key}: {energies[key] * HARTREE2KCAL} kcal/mol")

bond: 3.131132240161536 kcal/mol
angle: 0.6492496303146331 kcal/mol
torsion: 0.0 kcal/mol
bond_bond: -0.16173706689116085 kcal/mol
bond_angle: 0.3308725968699845 kcal/mol
angle_angle: 0.0 kcal/mol
torsion_bond: 0.0 kcal/mol
torsion_angle: 0.0 kcal/mol
torsion_angle_angle: 0.0 kcal/mol
perm_elec: -479.91535684551275 kcal/mol
pol: -100.86216706600197 kcal/mol
ct_direct: -152.77762350401704 kcal/mol
xpol: -6.466410472455853 kcal/mol
pauli: 595.70373021326 kcal/mol
disp: -121.22930767713305 kcal/mol
total: -261.59761795140565 kcal/mol


In [6]:
# Reference numbers from other branch
#'perm_elec': tensor(-479.9149),
#'pauli': tensor(595.7025),
#'disp': tensor(-121.2292),
#'elec_pol': tensor(-100.8528),
#'xpol': tensor(-22.1401),
#'pol': tensor(-122.9929),
#'pol_ct': tensor(-100.8621),
#'ct': tensor(-152.7866),
#'ct_direct': tensor(-152.7774),
#'ct_indirect': tensor(-0.0092),
#'total': tensor(-281.2212)